# 04. Huấn luyện mô hình

Bài toán chính: hồi quy dự đoán `G3`. Notebook này fit bộ tiền xử lý trên train, huấn luyện ít nhất ba mô hình, so sánh hai scenario có/không có `G1`, `G2`, chọn model bằng validation rồi đánh giá đúng một lần trên test.

Phần phân lớp đỗ/trượt là phần mở rộng; nhóm chỉ cần làm sau khi hoàn thành tốt hồi quy chính.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def find_project_root():
    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / 'data').exists() and (folder / 'src').exists():
            return folder
    raise FileNotFoundError('Không tìm thấy thư mục gốc của project.')

PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = ARTIFACTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
valid_df = pd.read_csv(PROCESSED_DIR / 'valid.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test.csv')
RANDOM_STATE = 42
print(f'Train: {train_df.shape}, valid: {valid_df.shape}, test: {test_df.shape}')

In [ ]:
def build_features(frame, use_previous_grades=True):
    result = frame.copy()
    result['absences_log1p'] = np.log1p(result['absences'])
    if use_previous_grades:
        result['previous_grade_mean'] = (result['G1'] + result['G2']) / 2
        result['previous_grade_gap'] = result['G2'] - result['G1']
    else:
        result = result.drop(columns=['G1', 'G2'])
    return result

def make_data(frame, use_previous_grades=True):
    engineered = build_features(frame, use_previous_grades)
    X = engineered.drop(columns=['G3'])
    y = engineered['G3'].astype(float)
    return X, y

def make_preprocessor(X):
    numeric_features = X.select_dtypes(include='number').columns.tolist()
    categorical_features = X.select_dtypes(exclude='number').columns.tolist()
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    try:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', encoder)
    ])
    return ColumnTransformer([
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features)
    ], remainder='drop', verbose_feature_names_out=True)

def regression_metrics(y_true, y_pred):
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'r2': float(r2_score(y_true, y_pred))
    }

In [ ]:
regression_factories = {
    'linear_regression': lambda: LinearRegression(),
    'random_forest': lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    'gradient_boosting': lambda: GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2, random_state=RANDOM_STATE)
}
scenarios = {'A_with_G1_G2': True, 'B_without_G1_G2': False}
validation_records = []
fitted_regression = {}

for scenario_name, use_previous_grades in scenarios.items():
    X_train, y_train = make_data(train_df, use_previous_grades)
    X_valid, y_valid = make_data(valid_df, use_previous_grades)
    for model_name, factory in regression_factories.items():
        pipeline = Pipeline([
            ('preprocessor', make_preprocessor(X_train)),
            ('model', factory())
        ])
        pipeline.fit(X_train, y_train)
        valid_pred = pipeline.predict(X_valid)
        metrics = regression_metrics(y_valid, valid_pred)
        validation_records.append({'scenario': scenario_name, 'model': model_name, **metrics})
        fitted_regression[(scenario_name, model_name)] = pipeline

validation_results = pd.DataFrame(validation_records).sort_values(['rmse', 'mae']).reset_index(drop=True)
validation_results.to_csv(ARTIFACTS_DIR / 'validation_results.csv', index=False)
display(validation_results)

In [ ]:
# Tinh chỉnh Random Forest bằng cross-validation trên train.
# Validation vẫn được giữ riêng để chọn model và test chỉ dùng ở cuối.
tune_scenario = 'A_with_G1_G2'
X_train_tune, y_train_tune = make_data(train_df, scenarios[tune_scenario])
X_valid_tune, y_valid_tune = make_data(valid_df, scenarios[tune_scenario])
rf_pipeline = Pipeline([
    ('preprocessor', make_preprocessor(X_train_tune)),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])
param_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [None, 12],
    'model__min_samples_leaf': [1, 2]
}
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=3,
    n_jobs=-1,
    refit=True
)
grid_search.fit(X_train_tune, y_train_tune)
tuned_valid_pred = grid_search.best_estimator_.predict(X_valid_tune)
tuned_metrics = regression_metrics(y_valid_tune, tuned_valid_pred)
validation_records.append({'scenario': tune_scenario, 'model': 'random_forest_grid_search', **tuned_metrics})
fitted_regression[(tune_scenario, 'random_forest_grid_search')] = grid_search.best_estimator_
validation_results = pd.DataFrame(validation_records).sort_values(['rmse', 'mae']).reset_index(drop=True)
validation_results.to_csv(ARTIFACTS_DIR / 'validation_results.csv', index=False)
print('Best parameters:', grid_search.best_params_)
display(validation_results)

In [ ]:
# Chọn model bằng validation RMSE, sau đó mới đánh giá trên test.
best_row = validation_results.iloc[0]
best_key = (best_row['scenario'], best_row['model'])
best_pipeline = fitted_regression[best_key]
best_use_previous_grades = scenarios[best_row['scenario']]
X_test, y_test = make_data(test_df, best_use_previous_grades)
test_pred = best_pipeline.predict(X_test)
test_metrics = regression_metrics(y_test, test_pred)

test_predictions = test_df[['subject', 'G3']].copy()
test_predictions['predicted_G3'] = test_pred
test_predictions['residual'] = test_predictions['G3'] - test_predictions['predicted_G3']
test_predictions['absolute_error'] = test_predictions['residual'].abs()
test_predictions['scenario'] = best_row['scenario']
test_predictions['model'] = best_row['model']
test_predictions.to_csv(ARTIFACTS_DIR / 'test_predictions.csv', index=False)

print(f'Model được chọn: {best_row["model"]}')
print(f'Scenario được chọn: {best_row["scenario"]}')
display(pd.Series(test_metrics, name='test_value').to_frame())

In [ ]:
# Lưu model, preprocessor, feature names và metrics để API dùng lại.
joblib.dump(best_pipeline, ARTIFACTS_DIR / 'best_regression_model.pkl')
joblib.dump(best_pipeline.named_steps['preprocessor'], ARTIFACTS_DIR / 'preprocessor.pkl')
feature_names = best_pipeline.named_steps['preprocessor'].get_feature_names_out().tolist()
(ARTIFACTS_DIR / 'feature_names.json').write_text(json.dumps(feature_names, ensure_ascii=False, indent=2), encoding='utf-8')

metrics_payload = {
    'task': 'regression',
    'target': 'G3',
    'best_model': str(best_row['model']),
    'best_scenario': str(best_row['scenario']),
    'validation_results': json.loads(validation_results.to_json(orient='records')),
    'test_metrics': test_metrics,
    'random_state': RANDOM_STATE
}
(ARTIFACTS_DIR / 'metrics.json').write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding='utf-8')

model_card = '\n'.join([
    '# Model card - Student Performance',
    '',
    '- Bài toán: hồi quy dự đoán điểm cuối kỳ G3.',
    f'- Model: {best_row["model"]}.',
    f'- Scenario: {best_row["scenario"]}.',
    f'- Test MAE: {test_metrics["mae"]:.4f}.',
    f'- Test RMSE: {test_metrics["rmse"]:.4f}.',
    f'- Test R2: {test_metrics["r2"]:.4f}.',
    '- Scenario A dùng G1, G2; Scenario B không dùng G1, G2.',
    '- Hạn chế: dữ liệu nhỏ và kết quả chỉ phản ánh các biến có trong Student Performance.'
])
(ARTIFACTS_DIR / 'model_card.md').write_text(model_card, encoding='utf-8')
print('Đã lưu model và artifact vào artifacts/.')

In [ ]:
# Biểu đồ chẩn đoán kết quả trên test.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, test_pred, alpha=0.65)
line_min = min(y_test.min(), test_pred.min())
line_max = max(y_test.max(), test_pred.max())
axes[0].plot([line_min, line_max], [line_min, line_max], linestyle='--', color='red')
axes[0].set_title('Điểm thật và điểm dự đoán')
axes[0].set_xlabel('G3 thật')
axes[0].set_ylabel('G3 dự đoán')
axes[1].hist(test_predictions['residual'], bins=20, edgecolor='black')
axes[1].set_title('Phân bố residual trên test')
axes[1].set_xlabel('G3 thật - G3 dự đoán')
axes[1].set_ylabel('Số dòng')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'regression_test_diagnostics.png', dpi=150)
plt.show()

## Ghi chú cho phần mở rộng classification

Nếu nhóm làm thêm classification, tạo nhãn `passed = G3 >= 10`, dùng Logistic Regression hoặc Random Forest Classifier và báo cáo Accuracy, Precision, Recall, F1, AUC-ROC. Không dùng phần mở rộng này để thay cho kết quả hồi quy chính của Đề 18.